# Mapa de Riesgo Espacial — Fase 2a (datos REALES de DataCrim)

**Qué hace**: cuenta delitos reales por zona de Lima y traduce ese conteo a un riesgo 0-100.
Genera `../src/models/predictor_v1.joblib` (una TABLA zona→riesgo, no un modelo que "piensa").

**Limitación honesta**: DataCrim solo trae el AÑO (sin hora). Por eso esto responde DÓNDE
hay más riesgo histórico, NO a qué hora. La hora será Fase 2b con datos propios de la app.

**Decisiones** (ver SDD design): tile 0.005° (~550m) · riesgo = percentile-rank del conteo
(robusto a zonas extremas) · conteo crudo sin per-cápita (sesgo documentado).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

pd.set_option('display.max_columns', None)
print('Setup OK')

## 1. Cargar datos reales

In [ ]:
# Usa la muestra; cambiar a datacrim_lima.csv cuando se baje el dataset completo.
CSV = Path('..') / 'data' / 'raw' / 'datacrim_lima_sample.csv'
raw = pd.read_csv(CSV)
# Tipos numericos + descartar filas sin coordenada
raw['lat'] = pd.to_numeric(raw['lat'], errors='coerce')
raw['lng'] = pd.to_numeric(raw['lng'], errors='coerce')
raw = raw.dropna(subset=['lat', 'lng']).reset_index(drop=True)
print('Incidentes:', len(raw))
raw.head()

## 2. EDA rápida

In [ ]:
print(raw['incident_generic'].value_counts().head(8), '\n')
print('Años:', sorted(raw['year'].dropna().unique().tolist()))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
raw['incident_generic'].value_counts().head(6).plot.barh(ax=ax[0]); ax[0].set_title('Top tipos de delito')
ax[1].scatter(raw['lng'], raw['lat'], s=3, alpha=0.2); ax[1].set_title('Puntos de delito (Lima)')
ax[1].set_xlabel('lng'); ax[1].set_ylabel('lat')
plt.tight_layout(); plt.show()

## 3. Binning espacial (cuadritos de 0.005°)

Cada delito cae en un cuadrito. `zone_id` lo identifica; el centro del cuadrito es la
posición estable de la zona (no la coordenada exacta del delito).


In [ ]:
LAT_MIN, LNG_MIN = -12.30, -77.20   # esquina del bbox de Lima
TILE = 0.005                          # ~550m

raw['gi'] = np.floor((raw['lat'] - LAT_MIN) / TILE).astype(int)
raw['gj'] = np.floor((raw['lng'] - LNG_MIN) / TILE).astype(int)
raw['zone_id'] = raw['gi'].astype(str) + '_' + raw['gj'].astype(str)
raw['zone_lat'] = LAT_MIN + (raw['gi'] + 0.5) * TILE
raw['zone_lng'] = LNG_MIN + (raw['gj'] + 0.5) * TILE
print('Zonas con actividad:', raw['zone_id'].nunique())

## 4. Conteo por zona + nombre de distrito

Contamos delitos por cuadrito. El distrito se resuelve desde un diccionario ubigeo→distrito
si está disponible; si no, queda "DESCONOCIDO" (las capas acumuladas no traen el nombre).


In [ ]:
# Tabla oficial ubigeo->distrito (INEI) + densidad de poblacion 2020
ref = pd.read_csv('../data/reference/ubigeo_peru.csv', dtype={'inei': str})
ref['inei'] = ref['inei'].str.zfill(6)
UBIGEO_TO_DIST = dict(zip(ref['inei'], ref['distrito']))
UBIGEO_TO_DIST['150101'] = 'CERCADO DE LIMA'  # evita confusion con Lima ciudad/provincia
print('ubigeos en tabla INEI:', len(UBIGEO_TO_DIST))

# Normalizar el ubigeo de los datos (puede venir como int o con .0)
raw['ubigeo'] = (raw['ubigeo'].astype(str)
                 .str.replace(r'\.0$', '', regex=True).str.zfill(6))

def zone_meta(group):
    ub = group['ubigeo'].mode()
    ub = ub.iloc[0] if len(ub) else None
    return pd.Series({'ubigeo': ub, 'district': UBIGEO_TO_DIST.get(ub, 'DESCONOCIDO')})

agg = raw.groupby('zone_id').agg(
    lat=('zone_lat', 'first'),
    lng=('zone_lng', 'first'),
    count=('zone_id', 'size'),
).reset_index()
meta = raw.groupby('zone_id').apply(zone_meta, include_groups=False).reset_index()
agg = agg.merge(meta, on='zone_id')
print('zonas con distrito conocido:', int((agg['district'] != 'DESCONOCIDO').sum()), '/', len(agg))
agg.sort_values('count', ascending=False).head()

## 5. riskScore 0-100 por RANKING

No dividimos por el máximo (una zona extrema aplastaría a todas a 0). Usamos el percentile-rank:
la posición de cada zona en el orden de conteos. "Top 10% más peligroso" → ~90.


In [ ]:
# rank pct: 0..1 segun posicion del conteo entre todas las zonas -> 0..100
agg['risk_score'] = (agg['count'].rank(method='average', pct=True) * 100).round().astype(int)

print(agg['risk_score'].describe().round(1))
print('\nTop 10 zonas de mayor riesgo:')
agg.sort_values('risk_score', ascending=False).head(10)[['zone_id', 'lat', 'lng', 'district', 'count', 'risk_score']]

## 6. Función de lookup (predict)

`predict(lat,lng)` ubica el cuadrito de la coordenada y devuelve su riesgo precalculado.
La hora se IGNORA (no existe en los datos). Fuera de cobertura → riesgo 0.


In [ ]:
def predict_risk(lat, lng, bundle):
    gi = int(np.floor((lat - bundle['lat_min']) / bundle['tile']))
    gj = int(np.floor((lng - bundle['lng_min']) / bundle['tile']))
    z = bundle['zones'].get(f'{gi}_{gj}')
    if z is None:
        return {'risk_score': 0, 'district': None, 'source': 'out_of_coverage'}
    return {'risk_score': int(z['risk_score']), 'district': z['district'], 'source': 'historical'}

print('definido')

## 7. Exportar artefacto (zones.csv + joblib)

In [ ]:
import json

HERE = Path.cwd()
ML_ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
MODELS_DIR = ML_ROOT / 'src' / 'models'
DATA_DIR = ML_ROOT / 'data' / 'processed'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

agg.to_csv(DATA_DIR / 'zones.csv', index=False)

# 1) zones.json -> puntos para el mapa de calor (liviano)
zones_json = [{'lat': round(float(r['lat']), 5), 'lng': round(float(r['lng']), 5),
               'risk': int(r['risk_score']), 'count': int(r['count']),
               'district': r['district']}
              for _, r in agg.iterrows()]
(DATA_DIR / 'zones.json').write_text(json.dumps(zones_json), encoding='utf-8')

# 2) districts.json -> ranking por distrito (riesgo = ranking del total de incidentes)
dist = (agg[agg['district'] != 'DESCONOCIDO'].groupby('district')
        .agg(count=('count', 'sum'), zones=('zone_id', 'size')).reset_index())
dist['risk'] = (dist['count'].rank(pct=True) * 100).round().astype(int)
dist = dist.sort_values('risk', ascending=False)
districts_json = [{'district': r['district'], 'risk': int(r['risk']),
                   'count': int(r['count']), 'zones': int(r['zones'])}
                  for _, r in dist.iterrows()]
(DATA_DIR / 'districts.json').write_text(json.dumps(districts_json, ensure_ascii=False), encoding='utf-8')

# 3) bundle joblib (lookup zona->riesgo)
zones = {r['zone_id']: {'lat': float(r['lat']), 'lng': float(r['lng']),
                        'district': r['district'], 'count': int(r['count']),
                        'risk_score': int(r['risk_score'])}
         for _, r in agg.iterrows()}
bundle = {'version': 'v1-spatial', 'tile': TILE, 'lat_min': LAT_MIN, 'lng_min': LNG_MIN, 'zones': zones}
joblib.dump(bundle, MODELS_DIR / 'predictor_v1.joblib')

# 4) types.json -> desglose por tipo de delito (dato real de DataCrim)
_t = raw['incident_generic'].value_counts().head(10)
types_json = [{'type': str(k).title(), 'count': int(v)} for k, v in _t.items()]
(DATA_DIR / 'types.json').write_text(json.dumps(types_json, ensure_ascii=False), encoding='utf-8')

print('Guardado: predictor_v1.joblib + zones.json + districts.json | zonas:', len(zones))
print('Top 5 distritos:', [(d['district'], d['risk']) for d in districts_json[:5]])

## Próximos pasos
1. Bajar dataset completo (`extract_datacrim.py` sin --max-per-layer) → reemplazar la muestra.
2. Tabla ubigeo→distrito de INEI para nombres reales (hoy "DESCONOCIDO").
3. Conectar `risk_predictor.py` a este bundle (Fase apply, etapa 3, con tests TDD).
4. (Diferido) escribir a la tabla `risk_zones` + router `/predict`.

**Sesgos a recordar**: conteo crudo (zonas densas salen altas) · más denuncias ≠ más crimen · mapa estático.
